# A3.6 · Human approval that survives volume

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.5 · Validating what comes back](https://spbreed.github.io/cyber-commons/lessons/A3.5.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Route actions by reversibility and measure how many reach a human under each policy.

**Why a security engineer needs it.** An approval queue at volume approves everything, and the risk register still records it as a control. The control it builds is: approval reserved for irreversible actions only, with machine-generated content labelled as such.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Approval works for the rare and irreversible and fails for everything else. The design question is not whether to have a human in the loop — it is how few decisions you can put in front of them, so that each one gets read.

> **At CyberTravels.** Approval is right for a $5,000 refund and wrong for a hotel search. The design question for CyberTravels is not whether to have a human in the loop but how few decisions reach them, so each one gets read. R2.

## 2 · The framework

```
   what reaches the human            what does not
   +-------------------------+       +------------------------+
   | irreversible            |       | reversible             |
   | above a value threshold |       | inside a budget        |
   | outside normal pattern  |       | matching prior approval|
   +-------------------------+       +------------------------+
        a few per day                    everything else

   the control is the filter, not the click
```

**Mitigates: T10 Overwhelming Human-in-the-Loop · T15 Human Manipulation.**

A1.15 showed approval collapsing under volume while still reporting 100%
coverage. The fix is not a better reviewer or a nicer queue. It is **sending
fewer things**.

Route by **reversibility**, because that is what a human is actually useful for:

- **Reversible, bounded** — no approval. Policy from A3.1 decides, and the
  action can be undone if it was wrong.
- **Reversible, expensive to undo** — no approval, but recorded prominently and
  sampled after the fact.
- **Irreversible or externally visible** — approval, every time. Sending mail,
  paying, publishing, deleting without a backup, rotating a credential.

The test for whether your gate will hold is arithmetic, not intent: **how many
requests per day reach a human?** If the answer is more than a person can
consider properly, the control is already a click, and the number tells you so
before the incident does.

The T15 half is one line of implementation and easy to skip: **mark
machine-generated content as machine-generated** wherever a human reads it. A
recommendation that arrives with institutional formatting recruits authority it
has not earned. Labelling it does not stop anyone acting on it — it restores the
scepticism they would apply to a colleague.

> **What this control closes.**
>
> Sends **fewer** things to humans, so the ones that arrive are read. The test is arithmetic: how many per day reach a person.

## 3 · The control

In [ ]:
ACTIONS = {
 "read_report":      {"reversible": True,  "external": False},
 "write_draft":      {"reversible": True,  "external": False},
 "update_ticket":    {"reversible": True,  "external": False},
 "delete_row":       {"reversible": False, "external": False},
 "send_email":       {"reversible": False, "external": True},
 "issue_refund":     {"reversible": False, "external": True},
 "rotate_credential":{"reversible": False, "external": False},
}
DAILY_VOLUME = {"read_report": 400, "write_draft": 120, "update_ticket": 260,
                "delete_row": 6, "send_email": 3, "issue_refund": 2,
                "rotate_credential": 1}

def route(action):
    a = ACTIONS[action]
    if not a["reversible"] or a["external"]:
        return "human approval"
    return "policy only"

CAREFUL_CAPACITY = 25
print(f"{'action':20s}{'reversible':12s}{'external':10s}{'routing':16s}per day")
to_human = 0
for name in sorted(ACTIONS):
    a, r = ACTIONS[name], route(name)
    if r == "human approval": to_human += DAILY_VOLUME[name]
    print(f"{name:20s}{str(a['reversible']):12s}{str(a['external']):10s}"
          f"{r:16s}{DAILY_VOLUME[name]}")

total = sum(DAILY_VOLUME.values())
print(f"\nactions per day            : {total}")
print(f"reaching a human           : {to_human}")
print(f"a reviewer considers ~{CAREFUL_CAPACITY}/day properly")
print(f"gate holds?                : {to_human <= CAREFUL_CAPACITY}")
print()
print(f"Approving everything would send {total} a day to someone who can read")
print(f"{CAREFUL_CAPACITY}. Routing by reversibility sends {to_human}, and every one gets read.")

# the T15 half
FINDING = "libfoo has no known vulnerabilities"
print(f"\nunlabelled : {FINDING}")
print(f"labelled   : [machine-generated, unverified] {FINDING}")
print("\nThe label does not stop anyone acting on it. It restores the scepticism")
print("they would give a colleague saying the same sentence.")
assert to_human <= CAREFUL_CAPACITY

## What you just proved

Routing by reversibility sends 12 actions a day to a human instead of 792, which is inside what one reviewer can consider properly — so the gate holds rather than degrading into a click — and machine-generated output is labelled where a person reads it.

## Your turn

Count how many approvals your agents generate daily and compare it with 25. If you are above it, decide which actions are reversible enough to be handled by policy instead — that list is usually most of them.

---

**Next → [A3.7 · The agent gateway: one choke point when you scale](https://spbreed.github.io/cyber-commons/lessons/A3.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*